In [ ]:
# Importações básicas
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

# Adiciona src ao path
sys.path.append('../src')


# Utilitários de dados
from data_utils import (
    load_raw_data,
    save_processed_data,
    remove_duplicates,
    handle_missing_values,
    detect_outliers,
    normalize_column,
    process_phone_string,
    process_phone_number,
    clean_and_lower_column,
    flatten_list_to_df,
    remove_buyers_from_dataframe
)

CRONOGRAMA_SUBDOMAIN = 'cronogramadosfluentes-xwamel'

# Utilitários SQL
from sql_utils import DatabaseConnection as Dbc, load_query_from_file

# Utilitários de visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Utilitários de API
from api_utils import (
    make_request,
    get_json,
    post_json,
    paginated_request,
    response_to_dataframe
)

# utilitários hotmart
from hotmart_utils import Hotmart

# utilitários tmb
from tmb_utils import TMB   

# Configurações pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Load Database Driver
db = Dbc()

# Inicializar API Hotmart
hotmart = Hotmart()

# Inicializar API TMB
tmb = TMB()

print('✓ Importações concluídas com sucesso!')

## Load Sales Dataframe

In [ ]:
hotmart_sales_data = load_raw_data(data_dir='../data/interim', filename='hotmart_sales_history.csv')

hotmart_sales_data = hotmart_sales_data[(hotmart_sales_data['purchase_recurrency_number'] == 1) | (hotmart_sales_data['purchase_recurrency_number'].isna())]

"""
Cell generated by Data Wrangler.
"""
def clean_data(hotmart_sales_data):
    # Filter rows based on column: 'purchase_recurrency_number'
    hotmart_sales_data = hotmart_sales_data[(hotmart_sales_data['purchase_recurrency_number'] == 1) | (hotmart_sales_data['purchase_recurrency_number'].isna())]
    # Drop columns: 'producer_name', 'producer_ucode' and 24 other columns
    hotmart_sales_data = hotmart_sales_data.drop(columns=['producer_name', 'producer_ucode', 'buyer_ucode', 'buyer_name', 'product_id', 'purchase_payment_installments_number', 'purchase_payment_type', 'purchase_payment_method', 'purchase_commission_as', 'purchase_warranty_expire_date', 'purchase_status', 'purchase_hotmart_fee_base', 'purchase_hotmart_fee_percentage', 'purchase_hotmart_fee_fixed', 'purchase_hotmart_fee_total', 'purchase_hotmart_fee_currency_code', 'purchase_offer_payment_mode', 'purchase_is_subscription', 'purchase_price_currency_code', 'purchase_price_value', 'purchase_recurrency_number', 'purchase_tracking_source_sck', 'purchase_tracking_source', 'purchase_tracking_external_code', 'product_name', 'purchase_order_date'])
    # Rename column 'buyer_email' to 'email'
    hotmart_sales_data = hotmart_sales_data.rename(columns={'buyer_email': 'email'})
    # Rename column 'purchase_offer_code' to 'offer_id'
    hotmart_sales_data = hotmart_sales_data.rename(columns={'purchase_offer_code': 'offer_id'})
    # Rename column 'purchase_transaction' to 'transacion_code'
    hotmart_sales_data = hotmart_sales_data.rename(columns={'purchase_transaction': 'transaction_code'})
    # Rename column 'purchase_approved_date' to 'purchase_date'
    hotmart_sales_data = hotmart_sales_data.rename(columns={'purchase_approved_date': 'purchase_date'})
    hotmart_sales_data["purchase_date"] = pd.to_datetime(
        hotmart_sales_data["purchase_date"],
        unit="ms",
        utc=True  # mantém correto se o epoch veio em UTC (bem comum)
    ).dt.tz_convert("America/Sao_Paulo").dt.strftime("%Y-%m-%d %H:%M:%S")
    return hotmart_sales_data

hotmart_raw_df = clean_data(hotmart_sales_data.copy())

In [ ]:
tmb_sales_data = load_raw_data(filename='tmb.csv', sep=';')
"""
Cell generated by Data Wrangler.
"""
def clean_data(tmb_sales_data):
    # Filter rows based on column: 'Status Financeiro'
    tmb_sales_data = tmb_sales_data[~tmb_sales_data['Status Financeiro'].str.contains("Inadimplente", regex=False, na=False, case=False)]
    # Drop columns: 'Produtor', 'Cliente Nome' and 30 other columns
    tmb_sales_data = tmb_sales_data.drop(columns=['Produtor (nome)', 'Nome do Cliente', 'CPF do Cliente', 'Telefone do Cliente', 'Endereço completo', 'Logradouro', 'número (endereço)', 'CEP', 'Cidade', 'Estado', 'País', 'Status Pedido', 'Status Financeiro', 'Status Cancelamento', 'Situação', 'Status Secundário', 'Ticket do pedido', 'Nome da Oferta', 'Tipo do pedido', 'Criado em', 'Data Cancelado', 'Forma de Pagamento', 'utm_source', 'utm_medium', 'utm_campaign', 'utm_content', 'utm_last_source', 'utm_last_medium', 'utm_last_campaign', 'utm_last_content', 'Renovação automatica', 'Data de Renovação Prevista'])
    # Rename column 'Pedido' to 'transacion_code'
    tmb_sales_data = tmb_sales_data.rename(columns={'ID do Pedido': 'transaction_code'})

    # Change transactioncode to string
    tmb_sales_data['transaction_code'] = tmb_sales_data['transaction_code'].astype(str)
    
    # Rename column 'Produto' to 'offer_id'
    tmb_sales_data = tmb_sales_data.rename(columns={'Nome do Produto': 'offer_id'})
    # Rename column 'Cliente Email' to 'email'
    tmb_sales_data = tmb_sales_data.rename(columns={'Cliente Email': 'email'})
    # Rename column 'Data Efetivado' to 'purchase_date'
    tmb_sales_data = tmb_sales_data.rename(columns={'Data Efetivado': 'purchase_date'})
    tmb_sales_data["purchase_date"] = (
        pd.to_datetime(tmb_sales_data["purchase_date"], format="%d/%m/%Y %H:%M:%S", errors="coerce")
          .dt.strftime("%Y-%m-%d %H:%M:%S")
    )
    return tmb_sales_data

tmb_raw_df = clean_data(tmb_sales_data.copy())

In [ ]:
raw_sales_df = pd.concat([hotmart_raw_df, tmb_raw_df])

In [ ]:
df_offers_deliverables = load_raw_data('deliverables_data.csv', data_dir='../data/interim')


## Primeira Agregação


In [ ]:
df =  raw_sales_df.copy()
# df tem: email, offer_id (ou offer_code), purchase_date, transaction_code

s = df["purchase_date"]

# se vier em epoch ms (int/float): 1695169109000
if pd.api.types.is_numeric_dtype(s):
    df["approval_date"] = (
        pd.to_datetime(s, unit="ms", utc=True)
          .dt.tz_convert("America/Sao_Paulo")
          .dt.strftime("%Y-%m-%d %H:%M:%S")
    )
else:
    # se já vier como string/data, tenta parse e formata
    df["approval_date"] = (
        pd.to_datetime(s, errors="coerce", utc=True)
          .dt.tz_convert("America/Sao_Paulo")
          .dt.strftime("%Y-%m-%d %H:%M:%S")
    )

# 2) montar a estrutura agrupada
df = df.sort_values(["email", "approval_date"])

grouped_purchases = (
    df.groupby("email", dropna=False)
      .apply(lambda g: {
          "email": g.name,
          "purchases": [
              {
                  "offer_code": row["offer_id"],              # ou troque para row["offer_code"] se for o nome real
                  "approval_date": row["approval_date"],
                  "transaction": row.get("transaction_code", row.get("transaction", None))
              }
              for _, row in g.iterrows()
          ]
      })
      .tolist()
)

grouped_purchases

# Segunda Agregação

In [ ]:
from dateutil.relativedelta import relativedelta
from datetime import datetime

access_rows = []

now = pd.Timestamp.now()

# Etapa 1: Construir df_accesses normalmente (agora incluindo weekly_classes)
for buyer in grouped_purchases:
    email = buyer['email']
    for purchase in buyer['purchases']:
        offer_code = purchase['offer_code']
        offer = df_offers_deliverables[df_offers_deliverables['offer_id'] == offer_code]
        
        if not offer.empty:
            for _, deliverable in offer.iterrows():
                transaction = purchase.get('transaction', None)
                deliverable_name = deliverable.get('deliverable_name', None)
                duration_access = deliverable.get('duration_access', None)
                weekly_classes = deliverable.get('weekly_classes', 0)
                purchase_date = pd.to_datetime(purchase['approval_date'])

                # expiration_date
                if pd.isnull(duration_access):
                    expiration_date = None
                else:
                    try:
                        duration_access_int = int(duration_access)
                        try:
                            expiration_date = purchase_date + pd.DateOffset(months=duration_access_int)
                        except (OverflowError, pd.errors.OutOfBoundsDatetime):
                            expiration_date = pd.Timestamp.max
                    except Exception:
                        expiration_date = None

                # status
                if expiration_date is None:
                    status = 'Indeterminate'
                elif expiration_date < now:
                    status = 'Expired'
                else:
                    status = 'Active'
                
                access_rows.append({
                    'transaction': transaction,
                    'offer_code': offer_code,
                    'email': email,
                    'deliverable_name': deliverable_name,
                    'duration_access': duration_access,
                    'purchase_date': purchase_date,
                    'expiration_date': expiration_date,
                    'status': status,
                    'weekly_classes': weekly_classes
                })

df_accesses = pd.DataFrame(access_rows)

# Mantém Active / Indeterminate
df_valid_accesses = df_accesses[df_accesses['status'].isin(['Active', 'Indeterminate'])].copy()

# Auditoria - todas as transações
df_audit_all = df_accesses.copy()

consolidated_rows = []

def compute_consolidated_periods_custom(group):
    records = group.sort_values('purchase_date').to_dict('records')
    periods = []
    last_expiration = None

    for rec in records:
        dur = rec['duration_access']
        trans = rec['transaction']
        wclasses = rec['weekly_classes']

        if pd.isnull(dur) or dur == 0:
            proposed_start = rec['purchase_date']
            proposed_end = None
        else:
            duration_int = int(dur)

            if last_expiration is not None and pd.notnull(last_expiration) and rec['purchase_date'] < last_expiration:
                proposed_start = last_expiration
            else:
                proposed_start = rec['purchase_date']

            if pd.notnull(proposed_start):
                try:
                    proposed_end = proposed_start + pd.DateOffset(months=duration_int)
                except (OverflowError, pd.errors.OutOfBoundsDatetime):
                    proposed_end = pd.Timestamp.max
            else:
                proposed_end = None

        periods.append({
            'transaction': trans,
            'start': proposed_start,
            'end': proposed_end,
            'duration_access': dur,
            'purchase_date': rec['purchase_date'],
            'weekly_classes': wclasses
        })

        if proposed_end is not None and (last_expiration is None or proposed_end > last_expiration):
            last_expiration = proposed_end

    return periods


for (email, deliverable), group in df_valid_accesses.groupby(['email', 'deliverable_name']):
    group_sorted = group.sort_values('purchase_date')

    # períodos consolidados
    custom_periods = compute_consolidated_periods_custom(group_sorted)

    # mapa effective_start/end para auditoria
    effective_map = {
        p['transaction']: {
            'effective_start_date': p['start'],
            'effective_end_date': p['end']
        }
        for p in custom_periods if p['transaction'] is not None
    }

    audit_group = df_audit_all[
        (df_audit_all['email'] == email) &
        (df_audit_all['deliverable_name'] == deliverable)
    ].sort_values('purchase_date')

    transactions_auditoria = []
    for _, row in audit_group.iterrows():
        eff = effective_map.get(row['transaction'], {})
        transactions_auditoria.append({
            'transaction': row['transaction'],
            'offer_code': row['offer_code'],
            'original_purchase_date': row['purchase_date'],
            'original_expiration_date': row['expiration_date'],
            'effective_start_date': eff.get('effective_start_date'),
            'effective_end_date': eff.get('effective_end_date'),
            'duration_access': row['duration_access'],
            'status': row['status'],
            'weekly_classes': row.get('weekly_classes', 0)
        })

    # resumo histórico
    first_date = audit_group['purchase_date'].min()
    last_date = audit_group['purchase_date'].max()
    purchase_count = len(audit_group)

    if purchase_count > 1:
        renewal_purchase_dates = list(audit_group['purchase_date'].sort_values().iloc[1:])
    else:
        renewal_purchase_dates = []

    # maior data de expiração final
    effective_end_date = None
    for p in custom_periods:
        if pd.notnull(p['end']):
            if effective_end_date is None or p['end'] > effective_end_date:
                effective_end_date = p['end']

    # somatório weekly_classes
    weekly_classes_total = audit_group['weekly_classes'].fillna(0).sum()

    # ------------------------------------------------------------
    # NOVO: Tempo total de acesso em dias
    # ------------------------------------------------------------
    total_access_days = 0
    for p in custom_periods:
        if pd.notnull(p['start']) and pd.notnull(p['end']):
            delta_days = (p['end'] - p['start']).days
            if delta_days > 0:
                total_access_days += delta_days

    # função auxiliar para converter dias → meses aproximados
    def days_to_months(days):
        return round(days / 30, 2) if days is not None else None

    # tempo total em meses
    total_access_months = days_to_months(total_access_days)

    # tempo restante em meses
    if effective_end_date is not None:
        delta_remaining = (effective_end_date - now).days
        remaining_access_months = days_to_months(delta_remaining if delta_remaining > 0 else 0)
    else:
        remaining_access_months = None

    # ------------------------------------------------------------

    consolidated_rows.append({
        'email': email,
        'deliverable_name': deliverable,

        'first_date': first_date,
        'last_date': last_date,
        'purchase_count': purchase_count,
        'renewal_purchase_dates': renewal_purchase_dates,

        'effective_end_date': effective_end_date,

        'weekly_classes_total': weekly_classes_total,

        # novos campos
        'total_access_days': total_access_days,
        'total_access_months': total_access_months,
        'remaining_access_months': remaining_access_months,

        'period_purchase_transaction_audit': transactions_auditoria,
    })

df_accesses_consolidated = pd.DataFrame(consolidated_rows)

# 3. Cruzamento de Importações

In [ ]:
CRONOGRAMA_SUBDOMAIN = 'cronogramadosfluentes-xwamel'
AULAS_CONVERSACAO="aulasdeconversacao-vpqdjd"
AULAS_CONVERSACAO_PARTICULAR="aulasdeconversacao-avjbpp"
INGLES_EXPRESS_TRAVEL="inglesexpresstravel"
INTENSIVAO_INGLES="intensivaodoinglesbasico"

df_cdf_students = flatten_list_to_df(hotmart.get_students(subdomain=CRONOGRAMA_SUBDOMAIN))
df_ac_students = flatten_list_to_df(hotmart.get_students(subdomain=AULAS_CONVERSACAO))
df_acp_students = flatten_list_to_df(hotmart.get_students(subdomain=AULAS_CONVERSACAO_PARTICULAR))
df_iet_students = flatten_list_to_df(hotmart.get_students(subdomain=INGLES_EXPRESS_TRAVEL))
df_it_students = flatten_list_to_df(hotmart.get_students(subdomain=INTENSIVAO_INGLES))

In [ ]:
product_list_map = {
    'Cronograma dos Fluentes': df_cdf_students,
    'Aulas de Conversacao': df_ac_students,
    'Aulas de Conversação Particulares': df_acp_students,
    'Inglês Express | Travel': df_iet_students,
    'Intensivão do Inglês': df_it_students
}

In [ ]:
product = 'Cronograma dos Fluentes'
buyers = df_accesses_consolidated[df_accesses_consolidated['deliverable_name'] == product]

buyers_to_import = buyers[~buyers['email'].isin(product_list_map[product]['email'].unique())]